# $\Lambda = 4, \lambda = 0.2$: ev_op_Hp15_2f

Notebook created by HLD for the work arXiv: 2503.13368 [quant-ph, hep-th]

In this notebook, we run the VQE experiments for bosonic SU(2) matrix model at $\Lambda = 4, \lambda = 0.2$ using the variant ev_op_Hp15_2f of EvolvedOperatorAnsatz with the a list of different Estimator seeds.

In [1]:
import sys
import time
sys.path.append('../../utility')
from vqe_run import *
from qc_ansatze import *
from L4_evop_func import *

In [2]:
l4_evop = L4_evop(2.0)

Min absolute value is 0.008975
Max absolute value is 2.89778
Mean absolute value is 0.25249
None
E_exact = 3.89548


In [3]:
[print(f'{l4_evop.ansatz_names[i]}: {l4_evop.ansatz_list[i].num_parameters}') for i in range(len(l4_evop.ansatz_list))]

ev_op_Hp15: 15
ev_op_Hp20: 20
ev_op_Hp25: 25
ev_op_Hp30: 30
ev_op_H40: 40
ev_op_Hp15_2f: 30
ev_op_Hp20_2f: 40
ev_op_Hp25_2f: 50
ev_op_Hp30_2f: 60
ev_op_Hp40_2f: 80


[None, None, None, None, None, None, None, None, None, None]

In [4]:
#l4_l02_evop.ansatz_list[1].decompose().draw(output = 'mpl')

In [5]:
H4q = l4_evop.H4q

# VQE individual runs

In [6]:
class VQE_experiment():
    def __init__(self, _seed, opt, _operator):
        self.seed = _seed
        self.iterations = 650
        algorithm_globals.random_seed = self.seed
        self.optimizer = opt(maxiter = self.iterations)
        self.operator = _operator
        #estimator
        self.noiseless_estimator = AerEstimator(run_options={"seed": self.seed, "shots": 1024},
                                    transpile_options={"seed_transpiler": self.seed},)
        self.counts = []
        self.values = []
    def store_intermediate_result(self, eval_count, parameters, mean, std):
        self.counts.append(eval_count)
        self.values.append(mean)
    
    def run_qve_w_specified_optimizer(self, ansatz):
        vqe = VQE(self.noiseless_estimator, ansatz, self.optimizer, callback=self.store_intermediate_result)
        result = vqe.compute_minimum_eigenvalue(operator=self.operator).eigenvalue.real
        print(f"VQE result: {result:.5f}")
        return result

# COBYLA

In [8]:
seed_list = [170, 225, 128, 88, 28, 11, 34]
print(f'{l4_evop.ansatz_names[5]}')
r_cobyla=[]
for seed in seed_list:
    print('------------------------')
    print(f'seed is now {seed}')
    vqe_exp = VQE_experiment(seed, COBYLA, H4q)
    t0 = time.time()
    result = vqe_exp.run_qve_w_specified_optimizer(l4_evop.ansatz_list[5])
    t1 = time.time()
    print(f'Time taken = {t1-t0}')
    r_cobyla.append(pd.DataFrame({f'{l4_evop.ansatz_names[5]}_s{seed}': vqe_exp.values}))

ev_op_Hp15_2f
------------------------
seed is now 170
VQE result: 4.10308
Time taken = 437.8085789680481
------------------------
seed is now 225
VQE result: 4.39604
Time taken = 498.0998840332031
------------------------
seed is now 128
VQE result: 3.79566
Time taken = 453.75280833244324
------------------------
seed is now 88
VQE result: 4.02968
Time taken = 513.0019371509552
------------------------
seed is now 28
VQE result: 3.65233
Time taken = 490.3022520542145
------------------------
seed is now 11
VQE result: 5.51291
Time taken = 440.53030705451965
------------------------
seed is now 34
VQE result: 3.57374
Time taken = 478.7156128883362


In [9]:
df1 = pd.concat([r_cobyla[i] for i in range(len(r_cobyla))], axis = 1)
df1.to_csv('l4_l20_ev_op_Hp15_2f_cobyla_seeds.csv')

In [10]:
seed_list = [170, 225, 128, 88,28, 11, 34]
print(f'{l4_evop.ansatz_names[5]}')
r_spsa=[]
for seed in seed_list:
    print('------------------------')
    print(f'seed is now {seed}')
    vqe_exp = VQE_experiment(seed, SPSA, H4q)
    t0 = time.time()
    result = vqe_exp.run_qve_w_specified_optimizer(l4_evop.ansatz_list[5])
    t1 = time.time()
    print(f'Time taken = {t1-t0}')
    r_spsa.append(pd.DataFrame({f'{l4_evop.ansatz_names[5]}_s{seed}': vqe_exp.values}))
    

ev_op_Hp15_2f
------------------------
seed is now 170
VQE result: 4.04601
Time taken = 2007.1504089832306
------------------------
seed is now 225
VQE result: 3.96740
Time taken = 2025.204929113388
------------------------
seed is now 128
VQE result: 4.46684
Time taken = 2074.0251739025116
------------------------
seed is now 88
VQE result: 3.93682
Time taken = 2066.4006509780884
------------------------
seed is now 28
VQE result: 4.18133
Time taken = 2045.9998621940613
------------------------
seed is now 11
VQE result: 4.13392
Time taken = 2029.8123061656952
------------------------
seed is now 34
VQE result: 4.17928
Time taken = 33127.83835697174


In [11]:
df2 = pd.concat([r_spsa[i] for i in range(len(r_spsa))], axis = 1)
df2.to_csv('l4_l20_ev_op_Hp15_2f_spsa_seeds.csv')